# 05b ML Fixed Embeddings — Group 2: Fine-Tuned (Log Target)

**Group 2.** Trains regressors on fine-tuned GNN embeddings using a reconstruction loss objective and temporal warm-starting. Target is `log_systemic_risk_label`.

| Dataset | Model | Dim | Notes |
|---|---|---|---|
| `graphsage_v2_32_srisk_dataset.parquet` | GraphSAGE v2 | 32 | Reconstruction loss; 32-dim bottleneck |
| `graphsage_v2_64_srisk_dataset.parquet` | GraphSAGE v2 | 64 | Reconstruction loss; 64-dim bottleneck |
| `graphsage_v2_128_srisk_dataset.parquet` | GraphSAGE v2 | 128 | Reconstruction loss; 128-dim |
| `node2vec_v2_32_srisk_dataset.parquet` | Node2Vec v2 | 32 | Structural random walks; 32-dim |
| `node2vec_v2_64_srisk_dataset.parquet` | Node2Vec v2 | 64 | Structural random walks; 64-dim |
| `node2vec_v2_128_srisk_dataset.parquet` | Node2Vec v2 | 128 | Structural random walks; 128-dim |

> Run `03_g2_ref.ipynb` first to generate the parquet files.

In [1]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

sys.path.insert(0, os.path.abspath('../..'))
from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option('display.max_columns', 200)
PROJECT_ROOT = Path().resolve().parents[1]
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Load Datasets

In [2]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]
TOP1_COLS    = ["model", "train_top1_mae", "validation_top1_mae", "train_top1_rmse", "validation_top1_rmse"]

df_sage_32, feature_cols_sage_32 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="graphsage_v2_32_srisk_dataset.parquet",
)
print(f"GraphSAGE v2 32:  {df_sage_32.shape}  —  {len(feature_cols_sage_32)} embedding cols")

df_sage_64, feature_cols_sage_64 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="graphsage_v2_64_srisk_dataset.parquet",
)
print(f"GraphSAGE v2 64:  {df_sage_64.shape}  —  {len(feature_cols_sage_64)} embedding cols")

df_sage_128, feature_cols_sage_128 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="graphsage_v2_128_srisk_dataset.parquet",
)
print(f"GraphSAGE v2 128: {df_sage_128.shape}  —  {len(feature_cols_sage_128)} embedding cols")

df_n2v_32, feature_cols_n2v_32 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="node2vec_v2_32_srisk_dataset.parquet",
)
print(f"Node2Vec v2 32:   {df_n2v_32.shape}  —  {len(feature_cols_n2v_32)} embedding cols")

df_n2v_64, feature_cols_n2v_64 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="node2vec_v2_64_srisk_dataset.parquet",
)
print(f"Node2Vec v2 64:   {df_n2v_64.shape}  —  {len(feature_cols_n2v_64)} embedding cols")

df_n2v_128, feature_cols_n2v_128 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="node2vec_v2_128_srisk_dataset.parquet",
)
print(f"Node2Vec v2 128:  {df_n2v_128.shape}  —  {len(feature_cols_n2v_128)} embedding cols")

GraphSAGE v2 32:  (145536, 37)  —  32 embedding cols
GraphSAGE v2 64:  (145536, 69)  —  64 embedding cols
GraphSAGE v2 128: (145536, 133)  —  128 embedding cols
Node2Vec v2 32:   (145536, 37)  —  32 embedding cols
Node2Vec v2 64:   (145536, 69)  —  64 embedding cols
Node2Vec v2 128:  (145536, 133)  —  128 embedding cols


In [3]:
trainer_sage_32  = ModelTrainer(df=df_sage_32,  feature_cols=feature_cols_sage_32,  target_col="log_systemic_risk_label")
trainer_sage_64  = ModelTrainer(df=df_sage_64,  feature_cols=feature_cols_sage_64,  target_col="log_systemic_risk_label")
trainer_sage_128 = ModelTrainer(df=df_sage_128, feature_cols=feature_cols_sage_128, target_col="log_systemic_risk_label")
trainer_n2v_32   = ModelTrainer(df=df_n2v_32,   feature_cols=feature_cols_n2v_32,   target_col="log_systemic_risk_label")
trainer_n2v_64   = ModelTrainer(df=df_n2v_64,   feature_cols=feature_cols_n2v_64,   target_col="log_systemic_risk_label")
trainer_n2v_128  = ModelTrainer(df=df_n2v_128,  feature_cols=feature_cols_n2v_128,  target_col="log_systemic_risk_label")

print("GraphSAGE v2 32  —", trainer_sage_32.train_df.shape,  trainer_sage_32.val_df.shape)
print("GraphSAGE v2 64  —", trainer_sage_64.train_df.shape,  trainer_sage_64.val_df.shape)
print("GraphSAGE v2 128 —", trainer_sage_128.train_df.shape, trainer_sage_128.val_df.shape)
print("Node2Vec v2 32   —", trainer_n2v_32.train_df.shape,   trainer_n2v_32.val_df.shape)
print("Node2Vec v2 64   —", trainer_n2v_64.train_df.shape,   trainer_n2v_64.val_df.shape)
print("Node2Vec v2 128  —", trainer_n2v_128.train_df.shape,  trainer_n2v_128.val_df.shape)

GraphSAGE v2 32  — (109152, 37) (18192, 37)
GraphSAGE v2 64  — (109152, 69) (18192, 69)
GraphSAGE v2 128 — (109152, 133) (18192, 133)
Node2Vec v2 32   — (109152, 37) (18192, 37)
Node2Vec v2 64   — (109152, 69) (18192, 69)
Node2Vec v2 128  — (109152, 133) (18192, 133)


## Define Models

In [4]:
candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}

list(candidate_models)

['Linear Regression',
 'Ridge',
 'MLP',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## Train — GraphSAGE v2 (32-dim)

In [5]:
trainer_sage_32.train_all(candidate_models)
display(trainer_sage_32.leaderboard()[DISPLAY_COLS])
trainer_sage_32.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,XGBoost,0.011448,0.172571,0.054116,0.475737
1,Gradient Boosting,0.009603,0.170835,0.048998,0.527905
2,Random Forest,0.005895,0.21157,0.033668,0.570098
3,MLP,0.015098,0.12902,0.066872,0.593577
4,Linear Regression,0.044417,0.219193,0.148396,0.69944
5,Ridge,0.043523,0.26697,0.140441,0.964542


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,XGBoost,0.279715,0.660799,0.358115,0.763486
1,Gradient Boosting,0.238819,0.633424,0.327928,0.753747
2,Random Forest,0.190083,0.629917,0.237852,0.744014
3,MLP,0.36977,0.791959,0.458911,1.004151
4,Linear Regression,1.074485,1.175704,1.228648,1.335117
5,Ridge,0.978227,1.042282,1.128433,1.193775


## Train — GraphSAGE v2 (64-dim)

In [6]:
trainer_sage_64.train_all(candidate_models)
display(trainer_sage_64.leaderboard()[DISPLAY_COLS])
trainer_sage_64.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.00625,0.222523,0.034899,0.47682
1,XGBoost,0.010921,0.228263,0.050283,0.517896
2,Linear Regression,0.049361,0.192877,0.167334,0.572025
3,Gradient Boosting,0.009661,0.26859,0.04884,0.644939
4,MLP,0.015993,0.24402,0.07086,0.671033
5,Ridge,0.048717,0.830449,0.136796,2.218784


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.203485,0.873904,0.250027,1.043844
1,XGBoost,0.259171,0.845472,0.332129,1.035575
2,Linear Regression,1.236747,1.431497,1.414873,1.623697
3,Gradient Boosting,0.240639,0.865335,0.329606,1.05561
4,MLP,0.413833,1.251387,0.50514,1.871943
5,Ridge,0.92158,1.097569,1.056782,1.232556


## Train — GraphSAGE v2 (128-dim)

In [7]:
trainer_sage_128.train_all(candidate_models)
display(trainer_sage_128.leaderboard()[DISPLAY_COLS])
trainer_sage_128.leaderboard()[TOP1_COLS]

c:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.048246573986944e-07.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Gradient Boosting,0.009368,0.155646,0.047469,0.368775
1,Random Forest,0.005856,0.185729,0.03262,0.396094
2,XGBoost,0.010722,0.181954,0.048701,0.420469
3,MLP,0.013535,0.122658,0.067209,0.423457
4,Linear Regression,0.045303,0.442719,0.147762,1.121708
5,Ridge,0.045238,1.425793,0.121285,4.18767


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Gradient Boosting,0.22698,0.946638,0.317558,1.127859
1,Random Forest,0.18831,0.897335,0.232278,1.09331
2,XGBoost,0.236659,1.055256,0.317694,1.244481
3,MLP,0.39838,1.270932,0.492404,1.454266
4,Linear Regression,1.054016,1.048856,1.210998,1.182657
5,Ridge,0.770805,1.282352,0.887104,1.624819


## Train — Node2Vec v2 (32-dim)

In [8]:
trainer_n2v_32.train_all(candidate_models)
display(trainer_n2v_32.leaderboard()[DISPLAY_COLS])
trainer_n2v_32.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.006513,0.029829,0.031676,0.127925
1,Gradient Boosting,0.014009,0.029078,0.066285,0.132301
2,XGBoost,0.012967,0.030237,0.060304,0.133325
3,MLP,0.025562,0.043669,0.07604,0.142115
4,Linear Regression,0.051969,0.075288,0.132416,0.189062
5,Ridge,0.051968,0.075287,0.132416,0.189063


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.17691,0.695298,0.218293,0.78331
1,Gradient Boosting,0.369948,0.750131,0.469559,0.837503
2,XGBoost,0.309572,0.754493,0.418872,0.838324
3,MLP,0.412716,0.798008,0.507641,0.877227
4,Linear Regression,0.853323,1.089829,0.976674,1.19963
5,Ridge,0.85333,1.089839,0.976682,1.199643


## Train — Node2Vec v2 (64-dim)

In [9]:
trainer_n2v_64.train_all(candidate_models)
display(trainer_n2v_64.leaderboard()[DISPLAY_COLS])
trainer_n2v_64.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.006554,0.030517,0.031413,0.12919
1,XGBoost,0.012114,0.029839,0.056141,0.130153
2,Gradient Boosting,0.010919,0.030121,0.052318,0.132206
3,MLP,0.026238,0.050451,0.068908,0.156249
4,Ridge,0.057169,0.087332,0.143434,0.214718
5,Linear Regression,0.057173,0.087337,0.143434,0.214719


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.174514,0.664437,0.214641,0.757744
1,XGBoost,0.277582,0.721737,0.377416,0.802635
2,Gradient Boosting,0.241806,0.736145,0.340179,0.825314
3,MLP,0.356629,0.801482,0.437697,0.894158
4,Ridge,0.941604,1.143669,1.07145,1.304449
5,Linear Regression,0.941574,1.143643,1.07142,1.304418


## Train — Node2Vec v2 (128-dim)

In [10]:
trainer_n2v_128.train_all(candidate_models)
display(trainer_n2v_128.leaderboard()[DISPLAY_COLS])
trainer_n2v_128.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,XGBoost,0.010772,0.028016,0.048785,0.122607
1,Gradient Boosting,0.012656,0.027661,0.060229,0.124853
2,Random Forest,0.006459,0.02977,0.031811,0.126363
3,MLP,0.022557,0.049368,0.058823,0.173679
4,Linear Regression,0.059738,0.090209,0.148913,0.234886
5,Ridge,0.060223,0.09072,0.148532,0.23521


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,XGBoost,0.227103,0.688706,0.312947,0.768611
1,Gradient Boosting,0.319839,0.699256,0.410415,0.77868
2,Random Forest,0.180633,0.650049,0.221077,0.73181
3,MLP,0.266439,0.733358,0.352014,0.82028
4,Linear Regression,0.992722,1.142512,1.133329,1.302969
5,Ridge,0.987949,1.139336,1.127724,1.299769


## Top 1% Leaderboards

In [11]:
print("GraphSAGE v2 (32-dim)");  display(trainer_sage_32.leaderboard()[TOP1_COLS])
print("GraphSAGE v2 (64-dim)");  display(trainer_sage_64.leaderboard()[TOP1_COLS])
print("GraphSAGE v2 (128-dim)"); display(trainer_sage_128.leaderboard()[TOP1_COLS])
print("Node2Vec v2 (32-dim)");   display(trainer_n2v_32.leaderboard()[TOP1_COLS])
print("Node2Vec v2 (64-dim)");   display(trainer_n2v_64.leaderboard()[TOP1_COLS])
print("Node2Vec v2 (128-dim)");  display(trainer_n2v_128.leaderboard()[TOP1_COLS])

GraphSAGE v2 (32-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,XGBoost,0.279715,0.660799,0.358115,0.763486
1,Gradient Boosting,0.238819,0.633424,0.327928,0.753747
2,Random Forest,0.190083,0.629917,0.237852,0.744014
3,MLP,0.36977,0.791959,0.458911,1.004151
4,Linear Regression,1.074485,1.175704,1.228648,1.335117
5,Ridge,0.978227,1.042282,1.128433,1.193775


GraphSAGE v2 (64-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.203485,0.873904,0.250027,1.043844
1,XGBoost,0.259171,0.845472,0.332129,1.035575
2,Linear Regression,1.236747,1.431497,1.414873,1.623697
3,Gradient Boosting,0.240639,0.865335,0.329606,1.05561
4,MLP,0.413833,1.251387,0.50514,1.871943
5,Ridge,0.92158,1.097569,1.056782,1.232556


GraphSAGE v2 (128-dim)


...

,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.17691,0.695298,0.218293,0.78331
1,Gradient Boosting,0.369948,0.750131,0.469559,0.837503
2,XGBoost,0.309572,0.754493,0.418872,0.838324
3,MLP,0.412716,0.798008,0.507641,0.877227
4,Linear Regression,0.853323,1.089829,0.976674,1.19963
5,Ridge,0.85333,1.089839,0.976682,1.199643


Node2Vec v2 (64-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.174514,0.664437,0.214641,0.757744
1,XGBoost,0.277582,0.721737,0.377416,0.802635
2,Gradient Boosting,0.241806,0.736145,0.340179,0.825314
3,MLP,0.356629,0.801482,0.437697,0.894158
4,Ridge,0.941604,1.143669,1.07145,1.304449
5,Linear Regression,0.941574,1.143643,1.07142,1.304418


Node2Vec v2 (128-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,XGBoost,0.227103,0.688706,0.312947,0.768611
1,Gradient Boosting,0.319839,0.699256,0.410415,0.77868
2,Random Forest,0.180633,0.650049,0.221077,0.73181
3,MLP,0.266439,0.733358,0.352014,0.82028
4,Linear Regression,0.992722,1.142512,1.133329,1.302969
5,Ridge,0.987949,1.139336,1.127724,1.299769


## Hyperparameter Tuning

In [12]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([
        np.full(len(trainer.train_df), -1),
        np.zeros(len(trainer.val_df), dtype=int),
    ])
    search = RandomizedSearchCV(
        base_model, param_distributions,
        n_iter=n_iter, cv=PredefinedSplit(split_idx),
        scoring="neg_root_mean_squared_error",
        random_state=42, n_jobs=-1,
    )
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    return search.best_params_


RF_PARAMS = {
    "model__n_estimators":      [100, 200, 300],
    "model__max_depth":         [None, 5, 10],
    "model__min_samples_leaf":  [1, 2, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__max_features":      ["sqrt", "log2", 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    "model__max_iter":          [100, 200, 300],
    "model__max_depth":         [3, 5, 8, None],
    "model__learning_rate":     [0.005, 0.01, 0.05],
    "model__min_samples_leaf":  [5, 10, 20, 50, 100],
    "model__l2_regularization": [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__max_leaf_nodes":    [15, 20, 30, 40, 50, 60],
    "model__max_bins":          [64, 128, 255],
}
XGB_PARAMS = {
    "model__n_estimators":     [100, 200, 400],
    "model__max_depth":        [3, 4, 5, 6, 8, 10],
    "model__learning_rate":    [0.005, 0.01, 0.05],
    "model__subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "model__min_child_weight": [1, 2, 5, 10],
    "model__gamma":            [0, 0.1, 0.5, 1.0, 2.0],
    "model__reg_alpha":        [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__reg_lambda":       [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
}
MLP_PARAMS = {
    "model__hidden_layer_sizes": [(64,), (128,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    "model__activation":         ["relu", "tanh"],
    "model__alpha":              [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    "model__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    "model__learning_rate":      ["constant", "adaptive"],
    "model__batch_size":         [32, 64, 128, "auto"],
}

for label, t in [
    ("GraphSAGE v2 (32-dim)",  trainer_sage_32),
    ("GraphSAGE v2 (64-dim)",  trainer_sage_64),
    ("GraphSAGE v2 (128-dim)", trainer_sage_128),
    ("Node2Vec v2 (32-dim)",   trainer_n2v_32),
    ("Node2Vec v2 (64-dim)",   trainer_n2v_64),
    ("Node2Vec v2 (128-dim)",  trainer_n2v_128),
]:
    print(f"\n===== Tuning {label} =====")
    tune(t, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  "Random Forest (tuned)")
    tune(t, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  "Gradient Boosting (tuned)")
    tune(t, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, "XGBoost (tuned)")
    tune(t, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=5, random_state=42)), MLP_PARAMS, "MLP (tuned)")
    display(t.leaderboard()[DISPLAY_COLS])
    display(t.leaderboard()[TOP1_COLS])


===== Tuning GraphSAGE v2 (32-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP (tuned),0.023032,0.031975,0.080706,0.117463
1,XGBoost (tuned),0.030213,0.088471,0.126061,0.231813
2,Random Forest (tuned),0.027579,0.11555,0.126258,0.277181
3,Gradient Boosting (tuned),0.032794,0.096784,0.143851,0.27778
4,XGBoost,0.011448,0.172571,0.054116,0.475737
5,Gradient Boosting,0.009603,0.170835,0.048998,0.527905
6,Random Forest,0.005895,0.21157,0.033668,0.570098
7,MLP,0.015098,0.12902,0.066872,0.593577
8,Linear Regression,0.044417,0.219193,0.148396,0.69944
9,Ridge,0.043523,0.26697,0.140441,0.964542


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,MLP (tuned),0.470489,0.639553,0.571472,0.749987
1,XGBoost (tuned),0.990757,1.195867,1.093617,1.345376
2,Random Forest (tuned),0.938478,1.060477,1.068609,1.18851
3,Gradient Boosting (tuned),1.100067,1.191855,1.252014,1.371818
4,XGBoost,0.279715,0.660799,0.358115,0.763486
5,Gradient Boosting,0.238819,0.633424,0.327928,0.753747
6,Random Forest,0.190083,0.629917,0.237852,0.744014
7,MLP,0.36977,0.791959,0.458911,1.004151
8,Linear Regression,1.074485,1.175704,1.228648,1.335117
9,Ridge,0.978227,1.042282,1.128433,1.193775



===== Tuning GraphSAGE v2 (64-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP (tuned),0.037045,0.052908,0.147386,0.209606
1,XGBoost (tuned),0.036908,0.069232,0.160328,0.230844
2,Gradient Boosting (tuned),0.034103,0.082567,0.149139,0.235107
3,Random Forest (tuned),0.028747,0.10915,0.130138,0.267316
4,Random Forest,0.00625,0.222523,0.034899,0.47682
5,XGBoost,0.010921,0.228263,0.050283,0.517896
6,Linear Regression,0.049361,0.192877,0.167334,0.572025
7,Gradient Boosting,0.009661,0.26859,0.04884,0.644939
8,MLP,0.015993,0.24402,0.07086,0.671033
9,Ridge,0.048717,0.830449,0.136796,2.218784


...

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest (tuned),0.011011,0.028993,0.052825,0.127117
1,Random Forest,0.006554,0.030517,0.031413,0.12919
2,XGBoost (tuned),0.012383,0.030371,0.055305,0.129831
3,XGBoost,0.012114,0.029839,0.056141,0.130153
4,Gradient Boosting (tuned),0.014483,0.030023,0.069258,0.131197
5,MLP (tuned),0.017209,0.034684,0.063033,0.131302
6,Gradient Boosting,0.010919,0.030121,0.052318,0.132206
7,MLP,0.026238,0.050451,0.068908,0.156249
8,Ridge,0.057169,0.087332,0.143434,0.214718
9,Linear Regression,0.057173,0.087337,0.143434,0.214719


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest (tuned),0.294074,0.681497,0.372021,0.770344
1,Random Forest,0.174514,0.664437,0.214641,0.757744
2,XGBoost (tuned),0.283661,0.748313,0.377781,0.819182
3,XGBoost,0.277582,0.721737,0.377416,0.802635
4,Gradient Boosting (tuned),0.402512,0.746343,0.487667,0.823695
5,MLP (tuned),0.325291,0.710047,0.416308,0.795931
6,Gradient Boosting,0.241806,0.736145,0.340179,0.825314
7,MLP,0.356629,0.801482,0.437697,0.894158
8,Ridge,0.941604,1.143669,1.07145,1.304449
9,Linear Regression,0.941574,1.143643,1.07142,1.304418



===== Tuning Node2Vec v2 (128-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,XGBoost,0.010772,0.028016,0.048785,0.122607
1,Random Forest (tuned),0.010412,0.028044,0.05111,0.122683
2,Gradient Boosting,0.012656,0.027661,0.060229,0.124853
3,Gradient Boosting (tuned),0.014417,0.027815,0.067362,0.125038
4,XGBoost (tuned),0.011811,0.029179,0.052461,0.126177
5,Random Forest,0.006459,0.02977,0.031811,0.126363
6,MLP (tuned),0.020296,0.03631,0.065731,0.131094
7,MLP,0.022557,0.049368,0.058823,0.173679
8,Linear Regression,0.059738,0.090209,0.148913,0.234886
9,Ridge,0.060223,0.09072,0.148532,0.23521


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,XGBoost,0.227103,0.688706,0.312947,0.768611
1,Random Forest (tuned),0.286312,0.674034,0.362573,0.754165
2,Gradient Boosting,0.319839,0.699256,0.410415,0.77868
3,Gradient Boosting (tuned),0.38687,0.713283,0.468301,0.786189
4,XGBoost (tuned),0.268393,0.741292,0.354699,0.809336
5,Random Forest,0.180633,0.650049,0.221077,0.73181
6,MLP (tuned),0.328691,0.737422,0.423632,0.820989
7,MLP,0.266439,0.733358,0.352014,0.82028
8,Linear Regression,0.992722,1.142512,1.133329,1.302969
9,Ridge,0.987949,1.139336,1.127724,1.299769


## Comparison with Baseline Results

Reference results from `04_ML_Classic_Algorithms.ipynb` and `05_ML_GNN_Embeddings.ipynb`:

| Dataset | Best model | Train MAE | Val MAE | Train RMSE | Val RMSE |
|---|---|---|---|---|---|
| Classical features (DebtRank, PageRank, ...) | XGBoost (tuned) | 0.0068 | 0.0139 | 0.0409 | 0.0824 |
| GraphSAGE v1 (link prediction, 64-dim) | XGBoost (tuned) | 0.0110 | 0.0351 | 0.0639 | 0.2275 |
| Node2Vec v1 (64-dim) | XGBoost (tuned) | 0.0130 | 0.0367 | 0.0623 | 0.2050 |

Fixed embeddings (this notebook) — best model per dataset:

In [13]:
def best_row(trainer, label):
    row = trainer.leaderboard().iloc[0]
    return {
        "Dataset": label,
        "Best model": row["model"],
        "Train MAE": round(float(row["train_mae"]), 4),
        "Val MAE":   round(float(row["validation_mae"]), 4),
        "Train RMSE": round(float(row["train_rmse"]), 4),
        "Val RMSE":   round(float(row["validation_rmse"]), 4),
    }

comparison = pd.DataFrame([
    best_row(trainer_sage_32,  "GraphSAGE v2 (reconstruction, 32-dim)"),
    best_row(trainer_sage_64,  "GraphSAGE v2 (reconstruction, 64-dim)"),
    best_row(trainer_sage_128, "GraphSAGE v2 (reconstruction, 128-dim)"),
    best_row(trainer_n2v_32,   "Node2Vec v2 (structural, 32-dim)"),
    best_row(trainer_n2v_64,   "Node2Vec v2 (structural, 64-dim)"),
    best_row(trainer_n2v_128,  "Node2Vec v2 (structural, 128-dim)"),
])

comparison.set_index("Dataset")

,Best model,Train MAE,Val MAE,Train RMSE,Val RMSE
Dataset,,,,,
"GraphSAGE v2 (reconstruction, 32-dim)",MLP (tuned),0.0230,0.0320,0.0807,0.1175
"GraphSAGE v2 (reconstruction, 64-dim)",MLP (tuned),0.0370,0.0529,0.1474,0.2096
"GraphSAGE v2 (reconstruction, 128-dim)",MLP (tuned),0.0169,0.0617,0.0617,0.2096
"Node2Vec v2 (structural, 32-dim)",Random Forest (tuned),0.0086,0.0293,0.0415,0.1262
"Node2Vec v2 (structural, 64-dim)",Random Forest (tuned),0.0110,0.0290,0.0528,0.1271
"Node2Vec v2 (structural, 128-dim)",XGBoost,0.0108,0.0280,0.0488,0.1226
